In [1]:
print('==> Preparing data.............................')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.datasets import MNIST, CIFAR10, CIFAR100, SVHN

import os
import torch
from torch import nn,optim
import torch.nn.functional as F

from torchvision import datasets, transforms

from time import perf_counter

import  numpy as np
import torch.utils.data as Data

from torch.utils.data import Dataset, DataLoader

class Safeman(Dataset):
    
    def __init__(self, data,targets):
        super(Safeman, self).__init__()
        self.data = data
        self.targets = targets
        
     
    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        img, target = self.data[idx], self.targets[idx]
        return img, target


        
        
        
class Safeman_Filter(Safeman):   

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                new_targets.append(known.index(targets[i]))
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)
        
        
class Safeman_FilterF(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                dd = known.index(targets[i])
                if dd == 0:
                    new_targets.append(0)
                else:
                    new_targets.append(1)                   
                    
                #new_targets.append(known.index(targets[i]))
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)     
        
        
        
        
class Safeman_FilterB(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        new_targets = []
        for i in range(len(targets)):
            if targets[i] in known:
                new_targets.append(0)
            else:
                new_targets.append(1)
        self.targets = np.array(new_targets)
        self.data = self.data

class Safeman_FilterC(Safeman):
    
    def __Filter__(self, trainknown):
        train_class_num=len(trainknown)
        for i in range(0,len(self.targets)) :
            if self.targets[i]>train_class_num:
                self.targets[i] = train_class_num
        self.data = self.data



        
def setup_seed(seed):

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(8)



known=[0, 1, 2,3,4,5,6,7,8,9]


num_class=len(known)

x_train = np.load('./data/sCX_train_kitsune3n28.npy')
x_test = np.load('./data/sCX_final_test_kitsune3n28.npy')
y_train = np.load('./data/sCy_train_kitsune3n28.npy')
y_test = np.load('./data/sCy__final_test_kitsune3n28.npy')

print(x_train.shape, x_test.shape, y_train.shape,y_test.shape)


train_dataset = Data.TensorDataset(torch.tensor(x_train), torch.tensor(y_train))
train_dataset.data = train_dataset.tensors[0]
train_dataset.targets = train_dataset.tensors[1]


test_dataset = Data.TensorDataset(torch.tensor(x_test), torch.tensor(y_test))
test_dataset.data = test_dataset.tensors[0]
test_dataset.targets = test_dataset.tensors[1]

labels=['0','1']


train_dataset.classes = labels
test_dataset.classes = labels

train_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}
test_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}


b_s=128

num_class=len(labels)

train_loaderxuanze = torch.utils.data.DataLoader(train_dataset, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)

test_loaderxuanze = torch.utils.data.DataLoader(test_dataset, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)

print("done!")



==> Preparing data.............................
(9000, 1, 28, 28) (1000, 1, 28, 28) (9000,) (1000,)
done!


In [2]:
testsetA_data=test_dataset.data
testsetA_targets=test_dataset.targets

In [3]:
x_train_tempte=[]
y_train_tempte=[]
for i in range(len(testsetA_targets)):
    y_train_tempte+=[int(testsetA_targets[i])]
    a = np.resize(testsetA_data[i], (100))
    x_train_tempte += [a]

In [4]:
x_test21, y_test21 = torch.Tensor(x_train_tempte), torch.Tensor(y_train_tempte)

print(x_test21.shape, y_test21.shape)

test_dataset2 = Data.TensorDataset(x_test21, y_test21)
test_dataset2.data = test_dataset2.tensors[0]
test_dataset2.targets = test_dataset2.tensors[1]


test_loader2 = torch.utils.data.DataLoader(
    test_dataset2, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real train Data2:', len(test_dataset2))

torch.Size([1000, 100]) torch.Size([1000])
Real train Data2: 1000


/tmp/ipykernel_1856281/4091330859.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  x_test21, y_test21 = torch.Tensor(x_train_tempte), torch.Tensor(y_train_tempte)


In [5]:
trainset=train_dataset

unique,counts = np.unique(trainset.targets,return_counts=True)
print(unique, counts)

[0 1] [7306 1694]


In [6]:
X_trainset_data=trainset.data
X_trainset_targets=trainset.targets

In [7]:
unique,counts = np.unique(X_trainset_targets,return_counts=True)
print(unique, counts)

[0 1] [7306 1694]


In [8]:
X_trainset_targets

tensor([1, 1, 1,  ..., 0, 0, 0])

In [9]:
y_train2=[]

count=[7306,6]
num_class=2
lists = [[] for i in range(num_class)]
y_train_temp=[]
x_train_temp=[]

In [10]:
X_trainset_data.shape

torch.Size([9000, 1, 28, 28])

In [11]:
X_trainset_targets.shape

torch.Size([9000])

In [12]:
for i in range(len(X_trainset_targets)):
    if len(lists[int(X_trainset_targets[i])])<count[int(X_trainset_targets[i])]:
        lists[int(X_trainset_targets[i])].append(int(X_trainset_targets[i]))   
        a = np.resize(X_trainset_data[i], (100))        
        x_train_temp += [a]
        y_train_temp+=[int(X_trainset_targets[i])]


In [13]:
int(X_trainset_targets[i])

0

In [14]:
len(x_train_temp)

7312

In [15]:
unique,counts = np.unique(y_train_temp,return_counts=True)
print(unique, counts)

[0 1] [7306    6]


In [16]:
len(x_train_temp)

7312

In [17]:
trainset.data.shape

torch.Size([9000, 1, 28, 28])

In [18]:
x_train2, y_train2 = torch.Tensor(x_train_temp), torch.Tensor(y_train_temp)

print(x_train2.shape, y_train2.shape)

train_dataset2 = Data.TensorDataset(x_train2, y_train2)
train_dataset2.data = train_dataset2.tensors[0]
train_dataset2.targets = train_dataset2.tensors[1]

train_loader2 = torch.utils.data.DataLoader(
    train_dataset2, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


torch.Size([7312, 100]) torch.Size([7312])


In [19]:
from torch.nn import Module
from torch import nn
import numpy as np
import torch
from torchvision.datasets import mnist
from torch.nn import CrossEntropyLoss
from torch.optim import SGD
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor


num_classB=2

# Build the network
class Modelnsl8(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(100, 64) 
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 64)
        self.fc4 = nn.Linear(64, num_classB) 
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return F.log_softmax(x, dim = 1)

In [20]:
train_loader2 = train_loader2
model = Modelnsl8()
sgd = SGD(model.parameters(), lr=1e-1)
loss_fn = CrossEntropyLoss()
all_epoch = 100

for current_epoch in range(all_epoch):
    model.train()
    for idx, (train_x, train_label) in enumerate(train_loader2):
        sgd.zero_grad()
        predict_y = model(train_x.float().view(-1, 100))
        loss = loss_fn(predict_y, train_label.long())
        if idx % 1000 == 0:
            print('idx: {}, loss: {}'.format(idx, loss.sum().item()))
        loss.backward()
        sgd.step()
    print("epoch i=",current_epoch)
    all_correct_num = 0
    all_sample_num = 0
    model.eval()
    for idx, (test_x, test_label) in enumerate(test_loader2):
        predict_y = model(test_x.float().view(-1, 100)).detach()
        predict_y = np.argmax(predict_y, axis=-1)
        current_correct_num = predict_y == test_label
        all_correct_num += np.sum(current_correct_num.numpy(), axis=-1)
        all_sample_num += current_correct_num.shape[0]
    acc = all_correct_num / all_sample_num
    print('accuracy: {:.3f}'.format(acc))
print("training end")


idx: 0, loss: 0.6852282881736755
epoch i= 0
accuracy: 0.811
idx: 0, loss: 0.004255981184542179
epoch i= 1
accuracy: 0.812
idx: 0, loss: 0.002104545710608363
epoch i= 2
accuracy: 0.817
idx: 0, loss: 0.0014411142328754067
epoch i= 3
accuracy: 0.811
idx: 0, loss: 0.0011130343191325665
epoch i= 4
accuracy: 0.818
idx: 0, loss: 0.0011237793369218707
epoch i= 5
accuracy: 0.811
idx: 0, loss: 0.0009914860129356384
epoch i= 6
accuracy: 0.814
idx: 0, loss: 0.0010810234816744924
epoch i= 7
accuracy: 0.818
idx: 0, loss: 0.046255797147750854
epoch i= 8
accuracy: 0.816
idx: 0, loss: 0.0008324196096509695
epoch i= 9
accuracy: 0.810
idx: 0, loss: 0.0009590208646841347
epoch i= 10
accuracy: 0.819
idx: 0, loss: 0.0006826134631410241
epoch i= 11
accuracy: 0.814
idx: 0, loss: 0.000546374823898077
epoch i= 12
accuracy: 0.814
idx: 0, loss: 0.0006688974681310356
epoch i= 13
accuracy: 0.815
idx: 0, loss: 0.0006193447043187916
epoch i= 14
accuracy: 0.825
idx: 0, loss: 0.0007762474124319851
epoch i= 15
accuracy: